In [20]:
# Install Unsloth first (must be before other packages)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install --no-deps trl peft accelerate bitsandbytes -q
!pip install wandb groq==0.13.0 httpx==0.27.2 jsonlines -q

print("✅ All packages installed")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
✅ All packages installed


In [21]:
import os
import json
import torch
import wandb
import jsonlines
import datetime
import random
from groq import Groq
from google.colab import userdata, files

# HuggingFace
from huggingface_hub import login as hf_login

# Unsloth
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

# Training
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset

# ── Load API Keys from Colab Secrets ──────────────────────────
GROQ_API_KEY  = userdata.get('GROQ_API_KEY')
WANDB_API_KEY = userdata.get('WANDB_API_KEY')
HF_TOKEN      = userdata.get('HF_TOKEN')

# Initialize clients
groq_client = Groq(api_key=GROQ_API_KEY)

# Login to HuggingFace
hf_login(token=HF_TOKEN)
print("✅ HuggingFace logged in")

# Login to W&B
wandb.login(key=WANDB_API_KEY)
print("✅ W&B logged in")

print("\n✅ All imports and auth ready")
print(f"   GPU: {torch.cuda.get_device_name(0)}")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


✅ HuggingFace logged in
✅ W&B logged in

✅ All imports and auth ready
   GPU: Tesla T4


In [22]:
# Upload nova_mock_db.json from Task 2
print("Upload nova_mock_db.json from Task 2...")
uploaded = files.upload()

with open("nova_mock_db.json", "r") as f:
    db = json.load(f)

products  = db["products"]
faqs      = db["faqs"]
customers = db["customers"]
orders    = db["orders"]

# Extract useful context for training data generation
skincare_products = [p for p in products if p["category"] == "skincare"]
makeup_products   = [p for p in products if p["category"] == "makeup"]
apparel_products  = [p for p in products if p["category"] == "apparel"]

print(f"✅ Real NOVA data loaded")
print(f"   Products  : {len(products)}")
print(f"   FAQs      : {len(faqs)}")
print(f"   Customers : {len(customers)}")
print(f"   Orders    : {len(orders)}")
print(f"\n   Skincare products : {len(skincare_products)}")
print(f"   Makeup products   : {len(makeup_products)}")
print(f"   Apparel products  : {len(apparel_products)}")

Upload nova_mock_db.json from Task 2...


Saving nova_mock_db.json to nova_mock_db (1).json
✅ Real NOVA data loaded
   Products  : 60
   FAQs      : 12
   Customers : 200
   Orders    : 500

   Skincare products : 10
   Makeup products   : 10
   Apparel products  : 10


In [23]:
# These rules are injected into every generation prompt
# So ALL 200 examples follow the exact same style guide

NOVA_BRAND_VOICE_RULES = """
NOVA BRAND VOICE RULES:
=========================
1. Always use "we" and "our" — speak as part of the NOVA team
2. Open with empathy or warmth — never jump straight to facts
3. Use positive framing even for bad news
   BAD:  "Your order is delayed"
   GOOD: "Your order is on its way — just taking a little longer than expected!"
4. Always end with a clear next step or offer of help
5. Maximum 3-4 sentences for routine replies
6. One emoji maximum — only where it feels natural 💛
7. Never say "I" — always "we"
8. Never sound robotic or corporate
9. Use contractions — "we're" not "we are", "you'll" not "you will"
10. If something went wrong, acknowledge it genuinely before solving it
"""

# Build real product + FAQ context strings for use in prompts
product_context = "\n".join([
    f"- {p['name']} (SKU: {p['product_id']}, "
    f"Category: {p['category']}, Price: ${p['price']})"
    for p in products[:20]  # First 20 as examples
])

faq_context = "\n".join([
    f"- Q: {f['question']}\n  A: {f['answer']}"
    for f in faqs
])

print("✅ Brand voice rules defined")
print(f"   Rules     : 10 style guidelines")
print(f"   Products  : {len(products)} real SKUs available")
print(f"   FAQs      : {len(faqs)} real Q&As available")

✅ Brand voice rules defined
   Rules     : 10 style guidelines
   Products  : 60 real SKUs available
   FAQs      : 12 real Q&As available


In [24]:
# We generate training data in batches of 20
# Each batch covers a different scenario type
# Real product names and FAQ answers are injected

SCENARIO_TYPES = [
    {
        "type"  : "order_delay",
        "count" : 30,
        "prompt": f"""Generate {{n}} training pairs for NOVA customer support.
Scenario: Customer is asking about a delayed order.
Use realistic order IDs like TRX-1042, TRX-2891.
Use real carriers: FedEx, UPS, DHL.

{NOVA_BRAND_VOICE_RULES}

Return ONLY valid JSON:
{{
  "pairs": [
    {{
      "instruction": "Respond to this customer support message as NOVA support",
      "input": "<generic robotic response to rewrite OR customer message>",
      "output": "<warm NOVA brand voice response>"
    }}
  ]
}}"""
    },
    {
        "type"  : "return_request",
        "count" : 25,
        "prompt": f"""Generate {{n}} training pairs for NOVA customer support.
Scenario: Customer wants to return a product.
Return policy: 30 days, free returns, unused products only.

{NOVA_BRAND_VOICE_RULES}

Return ONLY valid JSON:
{{
  "pairs": [
    {{
      "instruction": "Respond to this customer support message as NOVA support",
      "input": "<customer return request or generic response>",
      "output": "<warm NOVA brand voice response>"
    }}
  ]
}}"""
    },
    {
        "type"  : "product_inquiry",
        "count" : 30,
        "prompt": f"""Generate {{n}} training pairs for NOVA customer support.
Scenario: Customer asking about product ingredients or compatibility.
Use these real NOVA products:
{product_context}

{NOVA_BRAND_VOICE_RULES}

Return ONLY valid JSON:
{{
  "pairs": [
    {{
      "instruction": "Respond to this customer support message as NOVA support",
      "input": "<customer product question or generic response>",
      "output": "<warm NOVA brand voice response>"
    }}
  ]
}}"""
    },
    {
        "type"  : "faq_responses",
        "count" : 30,
        "prompt": f"""Generate {{n}} training pairs for NOVA customer support.
Scenario: Rewrite FAQ answers in NOVA brand voice.
Use these real FAQs as source material:
{faq_context}

{NOVA_BRAND_VOICE_RULES}

Return ONLY valid JSON:
{{
  "pairs": [
    {{
      "instruction": "Respond to this customer support message as NOVA support",
      "input": "<FAQ question or generic answer to rewrite>",
      "output": "<warm NOVA brand voice response>"
    }}
  ]
}}"""
    },
    {
        "type"  : "complaint_handling",
        "count" : 25,
        "prompt": f"""Generate {{n}} training pairs for NOVA customer support.
Scenario: Handling angry or frustrated customers.
These are HIGH stakes — empathy is critical.

{NOVA_BRAND_VOICE_RULES}

Return ONLY valid JSON:
{{
  "pairs": [
    {{
      "instruction": "Respond to this customer support message as NOVA support",
      "input": "<angry customer message or cold generic response>",
      "output": "<empathetic warm NOVA brand voice response>"
    }}
  ]
}}"""
    },
    {
        "type"  : "sizing_apparel",
        "count" : 25,
        "prompt": f"""Generate {{n}} training pairs for NOVA customer support.
Scenario: Customer asking about sizing for apparel or footwear.
Use these real NOVA products:
{product_context}

{NOVA_BRAND_VOICE_RULES}

Return ONLY valid JSON:
{{
  "pairs": [
    {{
      "instruction": "Respond to this customer support message as NOVA support",
      "input": "<sizing question or generic sizing response>",
      "output": "<warm NOVA brand voice response>"
    }}
  ]
}}"""
    },
    {
        "type"  : "shipping_tracking",
        "count" : 20,
        "prompt": f"""Generate {{n}} training pairs for NOVA customer support.
Scenario: Customer asking about shipping times or tracking.
Shipping: Standard 5-7 days, Express 2-3 days, International 10-14 days.

{NOVA_BRAND_VOICE_RULES}

Return ONLY valid JSON:
{{
  "pairs": [
    {{
      "instruction": "Respond to this customer support message as NOVA support",
      "input": "<shipping/tracking question or generic response>",
      "output": "<warm NOVA brand voice response>"
    }}
  ]
}}"""
    },
    {
        "type"  : "cancellation",
        "count" : 15,
        "prompt": f"""Generate {{n}} training pairs for NOVA customer support.
Scenario: Customer wants to cancel an order.
Policy: Can cancel within 2 hours of placement only.

{NOVA_BRAND_VOICE_RULES}

Return ONLY valid JSON:
{{
  "pairs": [
    {{
      "instruction": "Respond to this customer support message as NOVA support",
      "input": "<cancellation request or generic response>",
      "output": "<warm NOVA brand voice response>"
    }}
  ]
}}"""
    }
]

# Generate all training pairs
all_pairs = []
print("⏳ Generating training data using real NOVA data...\n")

for scenario in SCENARIO_TYPES:
    print(f"   Generating {scenario['count']} pairs — {scenario['type']}...")

    prompt = scenario["prompt"].replace("{n}", str(scenario["count"]))

    try:
        response = groq_client.chat.completions.create(
            model           = "llama-3.3-70b-versatile",
            messages        = [{"role": "user", "content": prompt}],
            max_tokens      = 4000,
            temperature     = 0.8,
            response_format = {"type": "json_object"}
        )

        data  = json.loads(response.choices[0].message.content)
        pairs = data.get("pairs", [])

        # Tag each pair with scenario type
        for pair in pairs:
            pair["scenario_type"] = scenario["type"]

        all_pairs.extend(pairs)
        print(f"   ✅ Got {len(pairs)} pairs — "
              f"total so far: {len(all_pairs)}")

    except Exception as e:
        print(f"   ⚠️  Error on {scenario['type']}: {e}")
        continue

print(f"\n✅ Training data generation complete")
print(f"   Total pairs generated : {len(all_pairs)}")

⏳ Generating training data using real NOVA data...

   Generating 30 pairs — order_delay...
   ✅ Got 29 pairs — total so far: 29
   Generating 25 pairs — return_request...
   ✅ Got 23 pairs — total so far: 52
   Generating 30 pairs — product_inquiry...
   ✅ Got 27 pairs — total so far: 79
   Generating 30 pairs — faq_responses...
   ✅ Got 32 pairs — total so far: 111
   Generating 25 pairs — complaint_handling...
   ✅ Got 24 pairs — total so far: 135
   Generating 25 pairs — sizing_apparel...
   ✅ Got 26 pairs — total so far: 161
   Generating 20 pairs — shipping_tracking...
   ✅ Got 20 pairs — total so far: 181
   Generating 15 pairs — cancellation...
   ✅ Got 15 pairs — total so far: 196

✅ Training data generation complete
   Total pairs generated : 196


In [25]:
import pandas as pd

# Validate all pairs have required fields
valid_pairs = []
for pair in all_pairs:
    if all(k in pair for k in ["instruction", "input", "output"]):
        if len(pair["output"]) > 20:    # Filter out empty/too short outputs
            valid_pairs.append(pair)

print(f"✅ Validation complete")
print(f"   Total pairs    : {len(all_pairs)}")
print(f"   Valid pairs    : {len(valid_pairs)}")
print(f"   Filtered out   : {len(all_pairs) - len(valid_pairs)}")

# Show distribution by scenario type
df = pd.DataFrame(valid_pairs)
print(f"\n📊 Distribution by scenario type:")
print(df["scenario_type"].value_counts().to_string())

# Preview 3 examples
print(f"\n📄 Sample training pairs:")
for i, pair in enumerate(valid_pairs[:3], 1):
    print(f"\n--- Example {i} ({pair['scenario_type']}) ---")
    print(f"INPUT  : {pair['input'][:100]}...")
    print(f"OUTPUT : {pair['output'][:150]}...")

# Save as JSONL
os.makedirs("data", exist_ok=True)
with jsonlines.open("data/brand_voice_train.jsonl", mode="w") as writer:
    for pair in valid_pairs:
        writer.write(pair)

print(f"\n✅ Saved to data/brand_voice_train.jsonl")
print(f"   {len(valid_pairs)} training examples ready")

✅ Validation complete
   Total pairs    : 196
   Valid pairs    : 171
   Filtered out   : 25

📊 Distribution by scenario type:
scenario_type
faq_responses         32
order_delay           29
product_inquiry       27
complaint_handling    24
return_request        23
shipping_tracking     20
cancellation          15
sizing_apparel         1

📄 Sample training pairs:

--- Example 1 (order_delay) ---
INPUT  : My order TRX-1042 is delayed, what's going on?...
OUTPUT : We're so sorry to hear that your order is taking a bit longer than expected! We're on it, and we'll make sure TRX-1042 gets to you as soon as possible...

--- Example 2 (order_delay) ---
INPUT  : I ordered TRX-2891 and it still hasn't arrived...
OUTPUT : We appreciate your patience, and we're happy to help you track down TRX-2891! It's on its way with FedEx, and we're working to get it to you ASAP. We'...

--- Example 3 (order_delay) ---
INPUT  : What's the status of my order TRX-4211?...
OUTPUT : We're excited for you to get 

In [26]:
# Model config
MODEL_NAME  = "unsloth/tinyllama-chat-bnb-4bit"
MAX_SEQ_LEN = 1024    # Keep short for Colab free tier
DTYPE       = None    # Auto-detect
LOAD_IN_4BIT = True   # QLoRA quantization

print(f"⏳ Loading {MODEL_NAME}...")
print(f"   Max sequence length : {MAX_SEQ_LEN}")
print(f"   4-bit quantization  : {LOAD_IN_4BIT}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    dtype         = DTYPE,
    load_in_4bit  = LOAD_IN_4BIT,
)

print(f"\n✅ Base model loaded")
print(f"   Model parameters : ~1.1B")
print(f"   Memory used      : "
      f"{torch.cuda.memory_allocated()/1e9:.2f} GB")

⏳ Loading unsloth/tinyllama-chat-bnb-4bit...
   Max sequence length : 1024
   4-bit quantization  : True
==((====))==  Unsloth 2026.3.17: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Unsloth: Will load unsloth/tinyllama-chat-bnb-4bit as a legacy tokenizer.



✅ Base model loaded
   Model parameters : ~1.1B
   Memory used      : 1.64 GB


In [27]:
# Apply LoRA adapters on top of the 4-bit model
model = FastLanguageModel.get_peft_model(
    model,
    r                   = 16,        # LoRA rank — higher = more capacity
    target_modules      = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha          = 32,        # Scaling factor
    lora_dropout        = 0.05,
    bias                = "none",
    use_gradient_checkpointing = True,
    random_state        = 42,
)

# Count trainable parameters
total_params    = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters()
                      if p.requires_grad)

print("✅ QLoRA adapters applied")
print(f"   Total parameters     : {total_params:,}")
print(f"   Trainable parameters : {trainable_params:,}")
print(f"   Trainable ratio      : "
      f"{100 * trainable_params / total_params:.2f}%")
print(f"   Memory used          : "
      f"{torch.cuda.memory_allocated()/1e9:.2f} GB")

✅ QLoRA adapters applied
   Total parameters     : 628,221,952
   Trainable parameters : 12,615,680
   Trainable ratio      : 2.01%
   Memory used          : 1.68 GB


In [28]:
# Format each training pair into chat format
# TinyLlama uses ChatML format

def format_training_example(pair: dict) -> str:
    """Convert a training pair to ChatML format."""
    return f"""<|system|>
You are NOVA's AI Support Assistant. Always respond in NOVA's warm,
empathetic brand voice. Use "we" and "our", be concise, end with a
helpful next step.</s>
<|user|>
{pair['instruction']}

Customer message or generic response to improve:
{pair['input']}</s>
<|assistant|>
{pair['output']}</s>"""


# Load from saved JSONL
train_pairs = []
with jsonlines.open("data/brand_voice_train.jsonl") as reader:
    for pair in reader:
        train_pairs.append(pair)

# Format all examples
formatted_texts = [format_training_example(p) for p in train_pairs]

# Split into train/eval (90/10)
random.shuffle(formatted_texts)
split_idx   = int(len(formatted_texts) * 0.9)
train_texts = formatted_texts[:split_idx]
eval_texts  = formatted_texts[split_idx:]

# Build HuggingFace datasets
train_dataset = Dataset.from_dict({"text": train_texts})
eval_dataset  = Dataset.from_dict({"text": eval_texts})

print("✅ Dataset formatted and split")
print(f"   Total examples : {len(formatted_texts)}")
print(f"   Train split    : {len(train_texts)}")
print(f"   Eval split     : {len(eval_texts)}")
print(f"\n📄 Sample formatted example:")
print("-" * 50)
print(formatted_texts[0][:400])
print("...")

✅ Dataset formatted and split
   Total examples : 171
   Train split    : 153
   Eval split     : 18

📄 Sample formatted example:
--------------------------------------------------
<|system|>
You are NOVA's AI Support Assistant. Always respond in NOVA's warm, 
empathetic brand voice. Use "we" and "our", be concise, end with a 
helpful next step.</s>
<|user|>
Respond to this customer support message as NOVA support

Customer message or generic response to improve:
Can I return a product without the original packaging?</s>
<|assistant|>
We understand that sometimes the origina
...


In [29]:
# Initialize W&B run
wandb.init(
    project = "nova-brand-voice",
    name    = "tinyllama-qlora-v1",
    config  = {
        "model"          : MODEL_NAME,
        "lora_r"         : 16,
        "lora_alpha"     : 32,
        "max_seq_length" : MAX_SEQ_LEN,
        "train_examples" : len(train_texts),
        "eval_examples"  : len(eval_texts),
        "epochs"         : 3,
        "batch_size"     : 2,
        "learning_rate"  : 2e-4,
    }
)

# Training arguments
training_args = TrainingArguments(
    output_dir              = "./nova-brand-voice",
    num_train_epochs        = 3,
    per_device_train_batch_size = 2,
    per_device_eval_batch_size  = 2,
    gradient_accumulation_steps = 4,
    warmup_steps            = 10,
    learning_rate           = 2e-4,
    fp16                    = not torch.cuda.is_bf16_supported(),
    bf16                    = torch.cuda.is_bf16_supported(),
    logging_steps           = 10,
    eval_strategy     = "steps",
    eval_steps              = 50,
    save_strategy           = "steps",
    save_steps              = 50,
    load_best_model_at_end  = True,
    report_to               = "wandb",    # W&B logging
    run_name                = "nova-brand-voice-v1",
    optim                   = "adamw_8bit",
    weight_decay            = 0.01,
    lr_scheduler_type       = "cosine",
    seed                    = 42,
)

# Build trainer
trainer = SFTTrainer(
    model           = model,
    tokenizer       = tokenizer,
    train_dataset   = train_dataset,
    eval_dataset    = eval_dataset,
    dataset_text_field = "text",
    max_seq_length  = MAX_SEQ_LEN,
    args            = training_args,
)

print("✅ Trainer configured")
print(f"   Epochs          : 3")
print(f"   Batch size      : 2 (effective: 8 with grad accum)")
print(f"   Learning rate   : 2e-4")
print(f"   W&B project     : nova-brand-voice")
print(f"\n⏳ Starting training...")
print(f"   Expected time   : 15-25 minutes on T4 GPU\n")

# Train!
trainer_stats = trainer.train()

print(f"\n✅ Training complete!")
print(f"   Training loss   : {trainer_stats.training_loss:.4f}")
print(f"   Training time   : "
      f"{trainer_stats.metrics['train_runtime']/60:.1f} minutes")

# Finish W&B run
wandb.finish()
print("✅ W&B run finished — check your dashboard!")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/153 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/18 [00:00<?, ? examples/s]

✅ Trainer configured
   Epochs          : 3
   Batch size      : 2 (effective: 8 with grad accum)
   Learning rate   : 2e-4
   W&B project     : nova-brand-voice

⏳ Starting training...
   Expected time   : 15-25 minutes on T4 GPU



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 153 | Num Epochs = 3 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 12,615,680 of 1,112,664,064 (1.13% trained)


Step,Training Loss,Validation Loss
50,0.351801,0.374924


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 


✅ Training complete!
   Training loss   : 0.7426
   Training time   : 1.4 minutes


eval/loss,▁
eval/runtime,▁
eval/samples_per_second,▁
eval/steps_per_second,▁
train/epoch,▁▂▄▅▇▇██
train/global_step,▁▂▄▅▇▇██
train/grad_norm,█▃▁▄▁▅
train/learning_rate,██▆▄▂▁
train/loss,█▂▁▁▁▁
eval/loss,0.37492
eval/runtime,1.5133


✅ W&B run finished — check your dashboard!


In [30]:
# Test prompts using real product names from Task 2
test_prompts = [
    {
        "scenario" : "Order delay",
        "input"    : f"Your order TRX-1042 is delayed by 3 days."
    },
    {
        "scenario" : "Return request",
        "input"    : f"Return request received for {skincare_products[0]['name']}."
    },
    {
        "scenario" : "Product question",
        "input"    : f"The {skincare_products[1]['name']} contains "
                     f"{', '.join(skincare_products[1].get('ingredients', ['Vitamin C'])[:2])}."
    },
    {
        "scenario" : "Angry customer",
        "input"    : "Customer complaint: order never arrived, very angry."
    },
    {
        "scenario" : "Shipping query",
        "input"    : "Standard shipping takes 5-7 business days."
    }
]

def generate_response(prompt_text: str, max_new_tokens: int = 150) -> str:
    """Generate response - bypasses Unsloth fast inference to avoid shape bug."""

    # Bypass Unsloth's fast inference engine entirely
    # Use standard HuggingFace generation instead
    from transformers import GenerationConfig

    model.eval()

    formatted = f"""<|system|>
You are NOVA's AI Support Assistant. Always respond in NOVA's warm,
empathetic brand voice.</s>
<|user|>
Respond to this customer support message as NOVA support:
{prompt_text}</s>
<|assistant|>"""

    inputs = tokenizer(
        formatted,
        return_tensors = "pt",
        truncation     = True,
        max_length     = 512
    ).to("cuda")

    # Disable Unsloth's custom attention by using standard generation config
    gen_config = GenerationConfig(
        max_new_tokens  = max_new_tokens,
        temperature     = 0.7,
        do_sample       = True,
        pad_token_id    = tokenizer.eos_token_id,
        eos_token_id    = tokenizer.eos_token_id,
        use_cache       = False,    # ← disable KV cache to avoid shape mismatch
    )

    with torch.no_grad():
        outputs = model.generate(
            input_ids       = inputs["input_ids"],
            attention_mask  = inputs["attention_mask"],
            generation_config = gen_config,
        )

    # Decode only new tokens
    input_length = inputs["input_ids"].shape[1]
    new_tokens   = outputs[0][input_length:]
    response     = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return response.strip()

print("✅ generate_response ready (Unsloth fast inference bypassed)")


# Score responses using Groq as judge
def score_response(response: str, scenario: str) -> dict:
    """Use Groq to score brand voice quality."""
    score_prompt = f"""Rate this NOVA customer support response on these criteria.
Scenario: {scenario}
Response: {response}

Score each from 1-5:
- warmth: How warm and empathetic is it?
- clarity: How clear and helpful is it?
- on_brand: How well does it follow brand voice (uses "we", ends with next step, not robotic)?

Return ONLY valid JSON:
{{"warmth": 1-5, "clarity": 1-5, "on_brand": 1-5, "overall": 1-5}}"""

    resp = groq_client.chat.completions.create(
        model           = "llama-3.3-70b-versatile",
        messages        = [{"role": "user", "content": score_prompt}],
        max_tokens      = 100,
        response_format = {"type": "json_object"}
    )
    return json.loads(resp.choices[0].message.content)


print("📊 EVALUATION — Fine-tuned Model vs Base Prompt")
print("=" * 65)

eval_results = []

for test in test_prompts:
    print(f"\n🧪 Scenario: {test['scenario']}")
    print(f"   Input: {test['input'][:80]}...")

    # Fine-tuned model response
    ft_response = generate_response(test["input"])
    ft_scores   = score_response(ft_response, test["scenario"])

    # Base Groq response (for comparison)
    base_resp = groq_client.chat.completions.create(
        model    = "llama-3.3-70b-versatile",
        messages = [
            {"role": "user",
             "content": f"Respond to this customer message: {test['input']}"}
        ],
        max_tokens = 150
    )
    base_response = base_resp.choices[0].message.content
    base_scores   = score_response(base_response, test["scenario"])

    print(f"\n   🤖 Base model response:")
    print(f"      {base_response[:120]}...")
    print(f"      Scores → warmth:{base_scores['warmth']} "
          f"clarity:{base_scores['clarity']} "
          f"on_brand:{base_scores['on_brand']}")

    print(f"\n   ✨ Fine-tuned response:")
    print(f"      {ft_response[:120]}...")
    print(f"      Scores → warmth:{ft_scores['warmth']} "
          f"clarity:{ft_scores['clarity']} "
          f"on_brand:{ft_scores['on_brand']}")

    eval_results.append({
        "scenario"      : test["scenario"],
        "input"         : test["input"],
        "base_response" : base_response,
        "ft_response"   : ft_response,
        "base_scores"   : base_scores,
        "ft_scores"     : ft_scores
    })

# Summary
print("\n" + "=" * 65)
print("📊 OVERALL SCORE COMPARISON")
print("=" * 65)

metrics = ["warmth", "clarity", "on_brand"]
for metric in metrics:
    base_avg = sum(r["base_scores"].get(metric, 0)
                   for r in eval_results) / len(eval_results)
    ft_avg   = sum(r["ft_scores"].get(metric, 0)
                   for r in eval_results) / len(eval_results)
    improvement = ((ft_avg - base_avg) / base_avg) * 100
    print(f"   {metric:12} → Base: {base_avg:.2f} | "
          f"Fine-tuned: {ft_avg:.2f} | "
          f"Improvement: +{improvement:.1f}%")

# Save evaluation results
with open("data/brand_voice_eval.json", "w") as f:
    json.dump({
        "timestamp"   : datetime.datetime.now().isoformat(),
        "model"       : MODEL_NAME,
        "eval_results": eval_results
    }, f, indent=2)

print("\n✅ Evaluation complete and saved")

Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ generate_response ready (Unsloth fast inference bypassed)
📊 EVALUATION — Fine-tuned Model vs Base Prompt

🧪 Scenario: Order delay
   Input: Your order TRX-1042 is delayed by 3 days....


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



   🤖 Base model response:
      Dear valued customer,

Re: Order TRX-1042

We regret to inform you that your order, TRX-1042, has been delayed by 3 days...
      Scores → warmth:4 clarity:5 on_brand:5

   ✨ Fine-tuned response:
      We're so sorry to hear that your order is delayed, and we appreciate that you're checking on TRX-1042. We're going to lo...
      Scores → warmth:4 clarity:4 on_brand:5

🧪 Scenario: Return request
   Input: Return request received for NOVA Vitamin C Serum....


Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



   🤖 Base model response:
      Dear valued customer,

Thank you for reaching out to us about returning your NOVA Vitamin C Serum. We apologize if the p...
      Scores → warmth:4 clarity:5 on_brand:5

   ✨ Fine-tuned response:
      We're glad you're interested in returning a product - we want you to get the best experience possible. We'll send you a ...
      Scores → warmth:4 clarity:5 on_brand:5

🧪 Scenario: Product question
   Input: The NOVA Hydrating Moisturizer contains Peptides, Vitamin C....


Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



   🤖 Base model response:
      The NOVA Hydrating Moisturizer sounds like a great product, with peptides to help stimulate collagen production and impr...
      Scores → warmth:4 clarity:5 on_brand:4

   ✨ Fine-tuned response:
      We're happy to help you get the most out of our Hydrating Moisturizer - it's packed with powerful ingredients to nourish...
      Scores → warmth:4 clarity:4 on_brand:5

🧪 Scenario: Angry customer
   Input: Customer complaint: order never arrived, very angry....


Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



   🤖 Base model response:
      Dear [Customer's Name],

I am so sorry to hear that your order never arrived. I can imagine how frustrating and disappoi...
      Scores → warmth:5 clarity:5 on_brand:4

   ✨ Fine-tuned response:
      We're sorry you're not happy with your order - we're here to help you get back on track. We're checking on the status of...
      Scores → warmth:4 clarity:4 on_brand:5

🧪 Scenario: Shipping query
   Input: Standard shipping takes 5-7 business days....

   🤖 Base model response:
      Thank you for your inquiry. Just to confirm, you can expect your order to arrive within 5-7 business days with our stand...
      Scores → warmth:4 clarity:5 on_brand:4

   ✨ Fine-tuned response:
      We're so sorry to hear that your order took a little longer than expected - we know how frustrating it can be. We apprec...
      Scores → warmth:4 clarity:4 on_brand:5

📊 OVERALL SCORE COMPARISON
   warmth       → Base: 4.20 | Fine-tuned: 4.00 | Improvement: +-4.8%
   clarity

In [31]:
# Get your HF username
from huggingface_hub import whoami
hf_username = whoami()["name"]
repo_name   = f"{hf_username}/nova-brand-voice-tinyllama"

print(f"⏳ Pushing model to HuggingFace Hub...")
print(f"   Repo: {repo_name}")

# Save model locally first
model.save_pretrained("nova-brand-voice")
tokenizer.save_pretrained("nova-brand-voice")

# Push to Hub
model.push_to_hub(repo_name, token=HF_TOKEN)
tokenizer.push_to_hub(repo_name, token=HF_TOKEN)

hf_link = f"https://huggingface.co/{repo_name}"

print(f"\n✅ Model pushed to HuggingFace!")
print(f"   🔗 Link: {hf_link}")
print(f"   Add this link to your README!")

⏳ Pushing model to HuggingFace Hub...
   Repo: Viddhula/nova-brand-voice-tinyllama


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 34.5kB / 50.5MB            

Saved model to https://huggingface.co/Viddhula/nova-brand-voice-tinyllama


No files have been modified since last commit. Skipping to prevent empty commit.



✅ Model pushed to HuggingFace!
   🔗 Link: https://huggingface.co/Viddhula/nova-brand-voice-tinyllama
   Add this link to your README!


In [32]:
files.download("data/brand_voice_train.jsonl")
files.download("data/brand_voice_eval.json")

print("✅ Files downloaded — push to GitHub:")
print("   → data/brand_voice_train.jsonl")
print("   → data/brand_voice_eval.json")
print("   → task4_finetune.ipynb  (File > Download > .ipynb)")
print(f"\n📌 Links to add to README:")
print(f"   W&B  : https://wandb.ai/YOUR_USERNAME/nova-brand-voice")
print(f"   HF   : {hf_link}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Files downloaded — push to GitHub:
   → data/brand_voice_train.jsonl
   → data/brand_voice_eval.json
   → task4_finetune.ipynb  (File > Download > .ipynb)

📌 Links to add to README:
   W&B  : https://wandb.ai/YOUR_USERNAME/nova-brand-voice
   HF   : https://huggingface.co/Viddhula/nova-brand-voice-tinyllama
